In [1]:
import pandas as pd
from tensorflow import keras


In [2]:
df= pd.read_csv('/kaggle/input/datasets/sabitaljuma/insurence-dataset/insurance_data.csv')
df.head(10)

,age,affordibility,bought_insurance
0,22,1,0
1,25,0,0
2,47,1,1
3,52,0,0
4,46,1,1
5,56,1,1
6,55,0,0
7,60,0,1
8,62,1,1
9,61,1,1


# Split train and test set

In [3]:
X= df.drop('bought_insurance', axis=1)
y=df['bought_insurance']

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(X,y, test_size= 0.2,random_state=42)

# Scaleing Data

**preprocessing: scaling the data so that both age and affordabilty in the same scale**

In [5]:
X_train_scaled= X_train.copy()
X_train_scaled['age']= X_train_scaled['age'] / 100

X_test_scaled= X_test.copy()
X_test_scaled['age']= X_test_scaled['age'] / 100

**Model Building: First we will build a model in keras/ tensorflow and see what weights and bias values comes up with. we will try to reproduce the same weights and bias in our plain python implementation of gradient descent.**

In [ ]:
# we will use one neuron, input shape 2 which is age and affordability. kernel and bias initialiser is weight and bias 
model= keras.Sequential([
    keras.layers.Dense(1,input_shape=(2,), activation='sigmoid', kernel_initializer= 'ones', bias_initializer= 'zeros')
])

model.compile(
    optimizer= 'adam',
    loss= 'binary_crossentropy',
    metrics= ['accuracy']
)
model.fit(X_train_scaled, y_train, epochs=5000)

Epoch 1/5000


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-09-11 00:02:59.869689: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 768ms/step - accuracy: 0.5000 - loss: 0.7428
Epoch 2/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5000 - loss: 0.7424
Epoch 3/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5000 - loss: 0.7420
Epoch 4/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.5000 - loss: 0.7416
Epoch 5/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5000 - loss: 0.7411
Epoch 6/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5000 - loss: 0.7407
Epoch 7/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5000 - loss: 0.7403
Epoch 8/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.5000 - loss: 0.7399
Epoch 9/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.5000 - loss: 0.7395
Epoch 10/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5000 - loss: 0.7390
Epoch 11/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5000 - loss: 0.7386
Epoch 12/5000
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.5000 - lo

> i had used epochs= 5000, then i got the accuracy: 0.9091, which is not bad at all. i think if we use more epochs it'll be better

**Evaluate the model on test set**

In [15]:
model.evaluate(X_test_scaled, y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 1.0000 - loss: 0.2647


[0.26470303535461426, 1.0]

In [16]:
model.predict(X_test_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


array([[0.81626713],
       [0.75767547],
       [0.8236742 ],
       [0.18819511],
       [0.38559136],
       [0.1959822 ]], dtype=float32)

In [17]:
y_test

9     1
25    1
8     1
21    0
0     0
12    0
Name: bought_insurance, dtype: int64

**Now get the value of weights and bias from the model**

In [18]:
coef, intercept= model.get_weights()
coef, intercept

(array([[5.0183096],
        [1.1966311]], dtype=float32),
 array([-2.7665412], dtype=float32))

> Here is showing w1, w2 and bias

In [19]:
import math
def sigmoid(x):
    return 1/ (1+math.exp(-x))
sigmoid(18)

0.9999999847700205

In [20]:
X_test

,age,affordibility
9,61,1
25,54,1
8,62,1
21,26,0
0,22,1
12,27,0


**Instead of model.predict, we are writing our own prediction function that uses w1, w2 and bias**

In [21]:
def predict_func(age, affordibility):
    weighted_sum= coef[0]*age + coef[1]*affordibility + intercept
    return sigmoid(weighted_sum)
predict_func(.27, 1)

/tmp/ipykernel_58/106242007.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return 1/ (1+math.exp(-x))


0.44646436703080217

In [22]:
predict_func(.18, 1)

/tmp/ipykernel_58/106242007.py:3: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return 1/ (1+math.exp(-x))


0.3392553710415985

In [48]:
import numpy as np
def sigmoid_numpy(x):
    return 1/(1+np.exp(-x))

sigmoid_numpy(np.array([12,0,1]))

array([0.99999386, 0.5       , 0.73105858])

**Created Log loss function**

In [49]:
def log_loss(y_true, y_predict):
    epsilon = 1e-15
    y_predict_new = [max(i,epsilon) for i in y_predict]
    y_predict_new = [min(i,1-epsilon) for i in y_predict_new]
    y_predict_new = np.array(y_predict_new)
    return -np.mean(y_true*np.log(y_predict_new)+(1-y_true)*np.log(1-y_predict_new))

# All right now comes the time to implement our final gradient descent function !! yay !!!

**im going to do it tooooo easy my friends. Donnorry!!**

In [69]:
def gradient_descent(age, affordability, y_true, epochs, loss_thresold):
    w1 = w2 = 1
    bias = 0
    rate = 0.5
    n = len(age)
    for i in range(epochs):
        weighted_sum = w1 * age + w2 * affordability + bias
        y_predicted = sigmoid_numpy(weighted_sum)
        loss = log_loss(y_true, y_predicted)

        w1d = (1/n)*np.dot(np.transpose(age),(y_predicted-y_true)) 
        w2d = (1/n)*np.dot(np.transpose(affordability),(y_predicted-y_true)) 

        bias_d = np.mean(y_predicted-y_true)
        w1 = w1 - rate * w1d
        w2 = w2 - rate * w2d
        bias = bias - rate * bias_d

        print (f'Epoch:{i}, w1:{w1}, w2:{w2}, bias:{bias}, loss:{loss}')

        if loss<=loss_thresold:
            break

    return w1, w2, bias

In [70]:
gradient_descent(X_train_scaled['age'], X_train_scaled['affordibility'], y_train, 1000, 0.46)

Epoch:0, w1:0.9736899318847281, w2:0.931388810977659, bias:-0.11748951666770448, loss:0.7428288579142563
Epoch:1, w1:0.9536535852311094, w2:0.8740290167758512, bias:-0.21881533456146035, loss:0.7072146449948488
Epoch:2, w1:0.9393731039296969, w2:0.8271852202997496, bias:-0.3053620401943441, loss:0.6814881914786812
Epoch:3, w1:0.9301932588998061, w2:0.7897792032048467, bias:-0.37884372361582785, loss:0.6633428084673968
Epoch:4, w1:0.9254091137248938, w2:0.7605726653866934, bias:-0.44108236820018304, loss:0.650742850709519
Epoch:5, w1:0.9243325693598607, w2:0.738313053647322, bias:-0.49384257986251556, loss:0.6420508089402462
Epoch:6, w1:0.926333296357235, w2:0.7218280753843739, bias:-0.5387319906498417, loss:0.6360356979531208
Epoch:7, w1:0.930858097563688, w2:0.7100747303660235, bias:-0.5771558825717441, loss:0.631816485354411
Epoch:8, w1:0.9374354910317362, w2:0.7021560855322683, bias:-0.6103083840841516, loss:0.6287844495353145
Epoch:9, w1:0.9456716791005845, w2:0.6973185496313956, b

(np.float64(6.9617150533861984),
 np.float64(1.3673304404593587),
 np.float64(-3.6920983113989734))

In [54]:
coef, intercept

(array([[5.0183096],
        [1.1966311]], dtype=float32),
 array([-2.7665412], dtype=float32))

**This shows that in the end we were able to come up with almost same value of w1,w2 and bias using a plain python implementation of gradient descent function**